# Soccer match outcome classification (H / D / A)

This notebook builds a **strictly pre-match** modeling table for **multi-class** outcomes (`H`=home win, `D`=draw, `A`=away win) using:

- **Rolling team form** from past matches only (no same-match goals/stats as features).
- **Closing odds** (`B365H/D/A`) transformed to **normalized implied probabilities**.

Models (shared **Stratified 5-fold CV**):

1. **Baseline**: Multinomial **Logistic Regression**
2. **SVM**: RBF `SVC` with `probability=True` (for log-loss)
3. **XGBoost**: `XGBClassifier` with `multi:softprob`



In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soccer_ml import build_model_table, ftr_to_int, load_raw_csvs

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)



## 1) Load data

If `data/raw/*.csv` is missing, run `python scripts/download_data.py` from the project root.


In [ ]:
raw_paths = sorted((ROOT / "data" / "raw").glob("*.csv"))
if not raw_paths:
    raise FileNotFoundError(
        "No CSVs in data/raw. Run: python scripts/download_data.py"
    )

matches = load_raw_csvs(raw_paths)
model_df = build_model_table(matches)
model_df.head()


## 2) EDA (dataset + target + key pre-match signals)


In [ ]:
model_df.shape


In [ ]:
model_df[["Div", "DateTime", "HomeTeam", "AwayTeam", "FTR", "y_ftr"]].head()


In [ ]:
vc = model_df["y_ftr"].value_counts().rename_axis("FTR").reset_index(name="count")
vc["percent"] = vc["count"] / vc["count"].sum()
vc


In [ ]:
fig, ax = plt.subplots()
order = ["H", "D", "A"]
counts = model_df["y_ftr"].value_counts().reindex(order)
ax.bar(counts.index.astype(str), counts.values, color=["#2ecc71", "#95a5a6", "#e74c3c"])
ax.set_title("Class balance (H/D/A)")
ax.set_ylabel("Matches")
plt.show()


In [ ]:
plot_cols = ["imp_h", "imp_d", "imp_a", "home_pts_roll", "away_pts_roll", "pts_diff_roll"]
sns.pairplot(
    model_df.sample(n=min(2000, len(model_df)), random_state=42),
    vars=plot_cols,
    hue="y_ftr",
    corner=True,
    plot_kws={"alpha": 0.35, "s": 18},
)
plt.suptitle("Pairplot (random subsample for speed)", y=1.02)
plt.show()


In [ ]:
corr = model_df[plot_cols].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation heatmap (selected numeric features)")
plt.show()


## 3) Modeling setup

We use the **same** `StratifiedKFold(5)` for every model and report **accuracy**, **macro/weighted F1**, and **negative log loss** (multiclass).


In [ ]:
NUMERIC_FEATURES = [
    "imp_h",
    "imp_d",
    "imp_a",
    "overround",
    "home_gf_roll",
    "home_ga_roll",
    "home_pts_roll",
    "home_home_share_roll",
    "away_gf_roll",
    "away_ga_roll",
    "away_pts_roll",
    "away_home_share_roll",
    "pts_diff_roll",
    "gf_diff_roll",
    "ga_diff_roll",
]
CAT_FEATURES = ["Div"]

X = model_df[NUMERIC_FEATURES + CAT_FEATURES]
y = ftr_to_int(model_df["y_ftr"])

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "logreg": Pipeline(
        steps=[
            ("prep", preprocess),
            (
                "clf",
                LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    solver="lbfgs",
                ),
            ),
        ]
    ),
    "svm_rbf": Pipeline(
        steps=[
            ("prep", preprocess),
            (
                "clf",
                SVC(
                    kernel="rbf",
                    class_weight="balanced",
                    probability=True,
                    random_state=42,
                ),
            ),
        ]
    ),
    "xgboost": Pipeline(
        steps=[
            ("prep", preprocess),
            (
                "clf",
                XGBClassifier(
                    objective="multi:softprob",
                    num_class=3,
                    n_estimators=400,
                    max_depth=5,
                    learning_rate=0.08,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    reg_lambda=1.0,
                    min_child_weight=1.0,
                    random_state=42,
                    n_jobs=1,
                    tree_method="hist",
                ),
            ),
        ]
    ),
}

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
    "neg_log_loss": "neg_log_loss",
}

rows = []
for name, est in models.items():
    out = cross_validate(est, X, y, cv=cv, scoring=scoring, n_jobs=1)
    row = {"model": name}
    for k, v in out.items():
        if k.startswith("test_"):
            metric = k.replace("test_", "")
            row[f"{metric}_mean"] = float(np.mean(v))
            row[f"{metric}_std"] = float(np.std(v))
    rows.append(row)

cv_results = pd.DataFrame(rows).set_index("model").sort_values("accuracy_mean", ascending=False)
cv_results


## 4) Confusion matrices (out-of-fold predictions, same CV)


In [ ]:
labels = ["H", "D", "A"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, est) in zip(axes, models.items()):
    y_hat = cross_val_predict(est, X, y, cv=cv, n_jobs=1)
    ConfusionMatrixDisplay.from_predictions(
        y,
        y_hat,
        display_labels=labels,
        normalize="true",
        values_format=".2f",
        ax=ax,
        colorbar=False,
    )
    ax.set_title(name)

plt.suptitle("Row-normalized confusion matrices (CV predictions)", y=1.02)
plt.show()


## 5) Example classification report (best CV accuracy model)


In [ ]:
best_name = cv_results["accuracy_mean"].idxmax()
best_est = models[best_name]
y_hat_best = cross_val_predict(best_est, X, y, cv=cv, n_jobs=1)
print("Best model:", best_name)
print(classification_report(y, y_hat_best, target_names=labels))


## 6) Notes for your course report

- **Leakage control**: rolling features use `shift(1)` after rolling means, computed per team timeline.
- **Odds** are closing prices; they encode public information — still legitimate **pre-kickoff** inputs, but discuss **market efficiency** and **ethical/betting** context.
- **Alternative validation**: chronological / season-based splits often better reflect deployment; we used stratified CV here for stable multi-class metrics across models.

